In [1]:
from google.colab import files
uploaded = files.upload()

Saving employees.csv to employees.csv
Saving logins.txt to logins.txt
Saving orders.json to orders.json
Saving sales.csv to sales.csv


In [2]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Pyspark Exercise").getOrCreate()
employees=spark.read.csv("employees.csv",header=True,inferSchema=True)
sales=spark.read.csv("sales.csv",header=True,inferSchema=True)
logins=spark.read.text("logins.txt")
orders=spark.read.json("orders.json")

employees.printSchema()
sales.printSchema()
logins.printSchema()
orders.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- city: string (nullable = true)

root
 |-- sale_id: integer (nullable = true)
 |-- emp_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)

root
 |-- value: string (nullable = true)

root
 |-- _corrupt_record: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- city: string (nullable = true)
 |-- customer: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- product: string (nullable = true)



In [22]:
logins.show()
logins.count()
unique_logins=logins.distinct()
unique_logins.show()
logins.groupBy("value").count().show()
logins.groupBy("value").count().orderBy("count",ascending=False).show(3)
logins.groupBy("value").count().filter("count>4").show()
login_counts = logins.groupBy("value").count()

login_dict = {}
for row in login_counts.collect():
    login_dict[row["value"]] = row["count"]
print(login_dict)


+-----+
|value|
+-----+
|Rahul|
|Sneha|
|Arjun|
|Rahul|
|Priya|
|Sneha|
|Rahul|
|Karan|
|Arjun|
|Sneha|
|Rahul|
| Amit|
|Priya|
|Karan|
|Sneha|
|Rahul|
|Meera|
|Arjun|
|Sneha|
|Rahul|
+-----+
only showing top 20 rows
+-----+
|value|
+-----+
|Meera|
|Sneha|
|Priya|
|Rahul|
|Arjun|
| Amit|
|Karan|
+-----+

+-----+-----+
|value|count|
+-----+-----+
|Meera|    1|
|Sneha|    6|
|Priya|    3|
|Rahul|    7|
|Arjun|    4|
| Amit|    2|
|Karan|    3|
+-----+-----+

+-----+-----+
|value|count|
+-----+-----+
|Rahul|    7|
|Sneha|    6|
|Arjun|    4|
+-----+-----+
only showing top 3 rows
+-----+-----+
|value|count|
+-----+-----+
|Sneha|    6|
|Rahul|    7|
+-----+-----+

{'Meera': 1, 'Sneha': 6, 'Priya': 3, 'Rahul': 7, 'Arjun': 4, 'Amit': 2, 'Karan': 3}


In [34]:
#EMPLOYEES EXERCISE
employees.show()
employees.count()
employees.filter(employees.department=="IT").show()
employees.filter(employees.salary>75000).show()
employees.groupBy().avg("salary").show()
employees.orderBy("salary",ascending=False).show(1)
employees.orderBy("salary").show(1)
employees.groupBy("department").count().show()
employees.groupBy("department").avg("salary").show()
employees.groupBy("city").count().show()
employees.orderBy("Salary",ascending=False).show(5)
employees.filter((employees.salary>70000) & (employees.city=="Hyderabad")).show()

+------+------+----------+------+---------+
|emp_id|  name|department|salary|     city|
+------+------+----------+------+---------+
|     1| Rahul|        IT| 70000|Hyderabad|
|     2| Sneha|        HR| 60000|Bangalore|
|     3| Arjun|        IT| 75000|  Chennai|
|     4| Priya|   Finance| 80000|Hyderabad|
|     5| Karan|        IT| 50000|   Mumbai|
|     6|  Amit|        HR| 58000|    Delhi|
|     7| Meera|   Finance| 82000|Bangalore|
|     8|  Ravi|        IT| 72000|Hyderabad|
|     9|  Neha|        HR| 61000|  Chennai|
|    10|Vikram|   Finance| 90000|    Delhi|
|    11| Anita|        IT| 65000|Bangalore|
|    12| Manoj|        HR| 62000|   Mumbai|
|    13| Divya|        IT| 77000|Hyderabad|
|    14|Sanjay|   Finance| 85000|  Chennai|
|    15| Pooja|        IT| 69000|Bangalore|
|    16| Kunal|        HR| 61000|    Delhi|
|    17| Sonal|   Finance| 88000|   Mumbai|
|    18|Deepak|        IT| 73000|Hyderabad|
|    19|  Ritu|        HR| 60000|  Chennai|
|    20| Akash|   Finance| 91000

In [45]:
#SALES EXERCISE
from pyspark.sql.functions import col
sales.show()
sales.count()
sales = sales.withColumn("revenue", col("quantity") * col("price"))
sales.show()
sales.groupBy().sum("revenue").show()
sales.groupBy("product").sum("revenue").show()
sales.groupBy("product").sum("quantity").show()
sales.groupBy("product").sum("quantity").orderBy("sum(quantity)", ascending=False).show(1)
sales.groupBy("emp_id").sum("revenue").orderBy("sum(revenue)", ascending=False).show(1)
sales.groupBy().avg("revenue").show()
sales.groupBy("product").sum("revenue").filter("sum(revenue) > 100000").show()


+-------+------+--------+--------+-----+-------+
|sale_id|emp_id| product|quantity|price|revenue|
+-------+------+--------+--------+-----+-------+
|      1|     1|  Laptop|       1|75000|  75000|
|      2|     2|   Mouse|       3|  500|   1500|
|      3|     3|Keyboard|       2| 1500|   3000|
|      4|     1| Monitor|       1|12000|  12000|
|      5|     4|  Laptop|       1|75000|  75000|
|      6|     3|   Mouse|       2|  500|   1000|
|      7|     5|Keyboard|       1| 1500|   1500|
|      8|     1|  Laptop|       1|75000|  75000|
|      9|     6|   Mouse|       4|  500|   2000|
|     10|     7| Monitor|       2|12000|  24000|
|     11|     8|Keyboard|       2| 1500|   3000|
|     12|     9|   Mouse|       3|  500|   1500|
|     13|    10|  Laptop|       1|75000|  75000|
|     14|    11| Monitor|       1|12000|  12000|
|     15|    12|Keyboard|       2| 1500|   3000|
|     16|    13|  Laptop|       1|75000|  75000|
|     17|    14|   Mouse|       3|  500|   1500|
|     18|    15| Mon

In [57]:
#ORDERS EXERCISE
orders.show()
orders.count()
orders.groupBy().sum("amount").show()
orders.groupBy("customer").sum("amount").show()
orders.groupBy("customer").sum("amount").orderBy("sum(amount)", ascending=False).show(1)
orders.groupBy("product").sum("amount").show()
orders.filter(orders.city == "Hyderabad").show()
orders.filter(orders.amount > 10000).show()
orders.groupBy("city").count().show()

+---------------+------+---------+--------+--------+--------+
|_corrupt_record|amount|     city|customer|order_id| product|
+---------------+------+---------+--------+--------+--------+
|              {|  NULL|     NULL|    NULL|    NULL|    NULL|
|    "orders": [|  NULL|     NULL|    NULL|    NULL|    NULL|
|           NULL| 75000|Hyderabad|   Rahul|       1|  Laptop|
|           NULL|  1500|Bangalore|   Sneha|       2|   Mouse|
|           NULL|  3000|  Chennai|   Arjun|       3|Keyboard|
|           NULL| 75000|Hyderabad|   Priya|       4|  Laptop|
|           NULL| 12000|   Mumbai|   Karan|       5| Monitor|
|           NULL|  1000|Hyderabad|   Rahul|       6|   Mouse|
|           NULL| 75000|Bangalore|   Sneha|       7|  Laptop|
|           NULL|  3000|  Chennai|   Arjun|       8|Keyboard|
|           NULL|  2000|Hyderabad|   Priya|       9|   Mouse|
|           NULL| 12000|Hyderabad|   Rahul|      10| Monitor|
|              ]|  NULL|     NULL|    NULL|    NULL|    NULL|
|       

In [58]:
#FINAL COMBINED EXERCISE
final_df = employees.join(sales, on="emp_id")
final_df.show()
final_df.groupBy("name").sum("revenue").show()
final_df.groupBy("name").sum("revenue").orderBy("sum(revenue)", ascending=False).show(5)
final_df.groupBy("department").sum("revenue").orderBy("sum(revenue)", ascending=False).show(1)
final_df.groupBy("name").sum("revenue").write.csv("final_sales_report", header=True)

+------+------+----------+------+---------+-------+--------+--------+-----+-------+
|emp_id|  name|department|salary|     city|sale_id| product|quantity|price|revenue|
+------+------+----------+------+---------+-------+--------+--------+-----+-------+
|     1| Rahul|        IT| 70000|Hyderabad|      8|  Laptop|       1|75000|  75000|
|     1| Rahul|        IT| 70000|Hyderabad|      4| Monitor|       1|12000|  12000|
|     1| Rahul|        IT| 70000|Hyderabad|      1|  Laptop|       1|75000|  75000|
|     2| Sneha|        HR| 60000|Bangalore|      2|   Mouse|       3|  500|   1500|
|     3| Arjun|        IT| 75000|  Chennai|      6|   Mouse|       2|  500|   1000|
|     3| Arjun|        IT| 75000|  Chennai|      3|Keyboard|       2| 1500|   3000|
|     4| Priya|   Finance| 80000|Hyderabad|      5|  Laptop|       1|75000|  75000|
|     5| Karan|        IT| 50000|   Mumbai|      7|Keyboard|       1| 1500|   1500|
|     6|  Amit|        HR| 58000|    Delhi|      9|   Mouse|       4|  500| 